# Explainable AML Transaction-Risk Triage
### for SME and Corporate Banking — technical deck

Julius Pabular · Pillar 5 Capstone · released model `20260904T225142-0dc8f82-hgb`

> Educational decision-support prototype trained on synthetic PaySim data. Outputs are risk scores and review priorities that help human investigators decide what to review first. This system makes no fraud or AML determination and performs no automatic blocking, account closure, customer risk rating, or regulatory reporting. Results on synthetic data do not establish real-world detection effectiveness, fairness, or regulatory suitability.

## 1 · Framing

- **Decision:** which of a day's transactions should an investigator review first, given K = 200 reviews?
- **Task:** binary classification on `isFraud` (1 = simulated fraud), score used for **ranking**.
- **Primary metric:** PR-AUC. **Operational:** Recall@K per review period (24 steps). Secondary: Precision@K, ROC-AUC, Brier/ECE, confusion at the operating point.
- **Prediction time:** end-of-day batch triage (posted balances available); aggregates use only strictly earlier rows.
- Non-use: no blocking, account closure, customer rating, regulatory reporting, or AML determination.

## 2 · Data: PaySim (synthetic, CC BY-SA 4.0)

- 6,362,620 rows · 11 columns · 0 nulls · 8,213 positives (prevalence 0.129%) · positives only in TRANSFER and CASH_OUT
- Steps 1–743 (hourly); **volume swings three orders of magnitude while positives arrive at a near-constant ~250/day** (DQ-10)
- Artifacts recorded, not hidden: all 16 zero-amount rows are positives (DQ-03); positives have exact balance bookkeeping (DQ-05); merchants carry no balances (DQ-06)

![volume](../figures/eda/eda_04_volume_positives_over_time.png)

*Sources: `reports/data_quality.md`, DQ-01..DQ-13.*

## 3 · Split and leakage controls

| split | steps | rows | positives | prevalence |
|---|---|---|---|---|
| train | 1–408 | 5,987,417 | 4,589 | 0.077% |
| validation | 409–552 | 181,068 | 1,504 | 0.83% |
| test | 553–743 | 194,135 | 2,120 | 1.09% |

- Temporal split; validation and test share the low-volume regime so the operating point transfers
- Every fitted transform and estimator recorded as `fitted_on: ["train"]`; causal aggregates verified against brute force; leakage test suite
- **Test split scored exactly once** after the operating point was frozen (`test_access.json`: 0 re-evaluations, 1 audited re-freeze)

## 4 · Features, selection, PCA

- 21-feature registry with rationale and prediction-time availability; sets: `primary` (24 cols, incl. 5 post-transaction), `strict_pretx` (19), `posttx_ablation` (10), `selected` (13), `pca_variant`
- Selection on train only: MI top-12 ∩ L1-logistic non-zero = 10 columns → 9 registry features; dropped 14 (origin aggregates carry nothing)
- PCA (diagnostic): 9 components for 95% of variance; components not used by candidates

![heatmap](../figures/eda/eda_06_correlation_heatmap.png)

## 5 · Model comparison (validation → test)

| candidate [set] | val PR-AUC | test PR-AUC | test Recall@200 |
|---|---|---|---|
| **hgb [primary]** (selected) | 1.0000 | 1.0000 | 0.7568 |
| hgb [strict_pretx] | 1.0000 | 0.9995 | 0.7568 |
| balanced_rf [primary] | 1.0000 | 0.9997 | 0.7568 |
| logreg [primary] | 0.9987 | 0.9954 | 0.7568 |
| logreg [strict_pretx] | 0.2776 | 0.2908 | 0.3539 |
| rule comparator | 0.1555 | 0.1856 | 0.3101 |
| random ranking | 0.0084 | 0.0109 | 0.1012 |

- Trees separate the generator almost perfectly with or without post-transaction fields; the linear model needs them (interaction).
- Tuned with RandomizedSearchCV on a seeded 1M-row subsample; selection key: val PR-AUC → Recall@K → Brier → explainability → cost.

![pr](../figures/models/pr_curves_test.png)

## 6 · Capacity analysis at K = 200

- Every test period holds 240–280 positives > K, so **Recall@200 = 0.7568** is a capacity ceiling (pooled 95% CI [0.725, 0.787]); Precision@200 = 1.0000
- K = 300 catches every positive; Precision@300 = 0.895
- Operating point (validation only): raw-score threshold 0.9719; on test 2,115 TP, 5 FN, 0 FP; calibrated probability display-only
- Illustrative: 200 positives/day surfaced vs 83 (rule), 28 (random)

![capacity](../figures/models/capacity_curve_test.png)

## 7 · Explainability (SHAP, PDP/ICE, permutation)

- Explainer: TreeExplainer (exact), 1,000 background / 2,000 eval rows
- Top mean |SHAP|: `amount_bucket` 0.01, `amount_to_orig_balance_ratio` 0.20, `dest_balance_delta` 0.11, `dest_balance_inconsistent_flag` 0.01
- **Permutation importance: the model depends on two bookkeeping flags** (drops {ex['pdp_ice']['permutation_importance'][0]['mean_drop_in_pr_auc']:.2f} and {ex['pdp_ice']['permutation_importance'][1]['mean_drop_in_pr_auc']:.2f} in PR-AUC); everything else is redundant → artifact dominance, stated in the report
- Three local waterfalls with plain-language captions; PDP/ICE valid for all top-5 features

![shap](../figures/explain/shap_01_global_bar.png)

## 8 · Bias & Fairness Analysis

- Availability record from actual columns: **no sensitive attributes or proxies** → demographic fairness cannot be computed; the report says so and never relabels slices as fairness
- **Operational error-slice analysis** (type, amount band, origin-balance band, day): under K = 200 the queue defers small-value / low-balance positives (Recall@200 0.00 in the two lowest amount bands) while flagging them at threshold
- Limitations: imbalance handling, leakage controls, nil val→test gap = separability not generalisation, artifact reliance, non-transferability
- Mitigations with mechanism / owner / trigger; governance-controlled audit plan for any real use

![slice](../figures/fairness/slice_amount_band.png)

## 9 · Reproducibility and governance

- Python 3.11.12, pinned `requirements.txt`, seed 42, config-driven CLI (22 commands), notebooks call the package
- `reproduce-check`: two fresh refits **identical** (tolerance 0.0e+00); released bundle `20260904T225142-0dc8f82-hgb` with `pipeline.sha256`, config snapshot, model card
- CI: lint, secret scan, 127 tests (leakage, test-access guard, vocabulary), no data tracked
- Repo: https://github.com/joopabs/aml-risk-triage-capstone

## 10 · What was learned, and next steps

- PaySim is almost perfectly separable; **the honest deliverable is the method, the guards and the caveats**, not the score
- Next: pilot the ranking-under-capacity design on governed real data with the fairness audit; capacity-aware slot reservation; monitor feature reliance and slice error rates
- Optional Step 8/9 (local scoring API, GenAI usage record) documented if attempted

> Educational decision-support prototype trained on synthetic PaySim data. Outputs are risk scores and review priorities that help human investigators decide what to review first. This system makes no fraud or AML determination and performs no automatic blocking, account closure, customer risk rating, or regulatory reporting. Results on synthetic data do not establish real-world detection effectiveness, fairness, or regulatory suitability.